In [2]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, make_scorer
from catboost import CatBoostClassifier, Pool, EShapCalcType, EFeaturesSelectionAlgorithm
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

import optuna

warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
ss = pd.read_csv('solution_format.csv')

train = train.drop("ID", axis=1)
df = train.copy()

In [4]:
grp_1 = train.filter(regex="_EventInd|index_").columns.to_list()
grp_2 = train.filter(regex="_A|_F|_G").columns.to_list()
grp_3 = list(set(train.columns).difference(grp_1).difference(grp_2))

In [5]:
# df[grp_1]=df[grp_1].fillna(0)
# #df[grp_2]=df[grp_2].apply(str).fillna(df.mode().iloc[0])
# df[grp_2]=df[grp_2].apply(str)
# df[grp_3]=df[grp_3].fillna(df.mean())


df[grp_1]=df[grp_1].fillna(0)
df[grp_2]=df[grp_2].apply(str).fillna(df.mode().iloc[0])
df[grp_3]=df[grp_3].fillna(df.mean())

In [6]:
X = df.drop("Target", axis=1)
y = df.Target 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

In [7]:
cv=KFold(n_splits=5,shuffle=True,random_state=0)

In [8]:
# model=CatBoostRegressor(iterations=1000,learning_rate=0.05,depth=8)

# model = CatBoostClassifier(iterations=100, depth=8, learning_rate=0.1, loss_function='Logloss', custom_metric=['AUC'], random_seed=42)

model = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=8, loss_function='Logloss', custom_metric=['AUC'], random_seed=1234)                

scores=[]
for i,j in enumerate(cv.split(X,y)):
    train=Pool(
        data=X.iloc[j[0]],
        label=y.iloc[j[0]],
        cat_features=grp_2
    )
    eval=Pool(
        data=X.iloc[j[1]],
        label=y.iloc[j[1]],
        cat_features=grp_2
    )
    model.fit(
        train,
        eval_set=eval,
        verbose=False
    )
    
    y_pred=model.predict(X.iloc[j[1]])
    score=f1_score(y.iloc[j[1]], y_pred, average='binary')
    scores.append(score)
    print(f"Fold {i}\t:{score}\n")

print(f"CV_SCORES_mean: {np.mean(scores)}")
print(f"CV_SCORES_std: {np.std(scores)}")

Fold 0	:0.7416413373860181

Fold 1	:0.7725321888412018

Fold 2	:0.7378048780487805

Fold 3	:0.7453237410071942

Fold 4	:0.7593123209169054

CV_SCORES_mean: 0.75132289324002
CV_SCORES_std: 0.012853981954864025


In [9]:
train=Pool(
    data=X_train,
    label=y_train,
    cat_features=grp_2
)
eval=Pool(
    data=X_test,
    label=y_test,
    cat_features=grp_2
)
model.fit(
    train,
    eval_set=eval,
    verbose=False
)

### Optimization with Optuna

In [10]:
def objective(trial):
    params = {
        "subsample": trial.suggest_float("subsample", 0.5, 0.99),
        'od_wait': trial.suggest_int('od_wait', 10, 50, step=2),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.2, 0.99),
        "random_strength": trial.suggest_int("random_strength", 1, 10, step=1),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.5),
        "max_depth": trial.suggest_int("max_depth", 4, 15, step=1),
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000, step=1),
        'learning_rate': trial.suggest_loguniform("learning_rate", 0.01, 0.2),
        }

    model=CatBoostClassifier(**params)

    scores=[]
    for i,j in enumerate(cv.split(X,y)):
        train=Pool(
            data=X.iloc[j[0]],
            label=y.iloc[j[0]],
            cat_features=grp_2
        )
        eval=Pool(
            data=X.iloc[j[1]],
            label=y.iloc[j[1]],
            cat_features=grp_2
        )
        model.fit(
            train,
            eval_set=eval,
            verbose=False
        )
        
        y_pred=model.predict(X.iloc[j[1]])
        score=f1_score(y.iloc[j[1]], y_pred, average='binary')
        scores.append(score)

        return np.mean(scores)

In [11]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=300, n_jobs=-1, timeout=40000)

[I 2024-05-22 14:28:48,813] A new study created in memory with name: no-name-15634ac2-b0c9-4c55-add8-341b372c1a22
[I 2024-05-22 14:28:57,289] Trial 2 finished with value: 0.7179487179487181 and parameters: {'subsample': 0.8415540216479793, 'od_wait': 44, 'colsample_bylevel': 0.6068751029479258, 'random_strength': 4, 'l2_leaf_reg': 2.870629200524341, 'max_depth': 6, 'n_estimators': 248, 'learning_rate': 0.19152601415442932}. Best is trial 2 with value: 0.7179487179487181.
[I 2024-05-22 14:29:37,223] Trial 3 finished with value: 0.7266982622432859 and parameters: {'subsample': 0.8566269911163171, 'od_wait': 18, 'colsample_bylevel': 0.8766531997827247, 'random_strength': 1, 'l2_leaf_reg': 9.76513557735635, 'max_depth': 8, 'n_estimators': 325, 'learning_rate': 0.0584843991720038}. Best is trial 3 with value: 0.7266982622432859.
[I 2024-05-22 14:30:45,895] Trial 4 finished with value: 0.7225433526011561 and parameters: {'subsample': 0.7663233326010428, 'od_wait': 18, 'colsample_bylevel': 0.

In [12]:
optuna.visualization.plot_optimization_history(study)

In [13]:
study.best_params

{'subsample': 0.7191808818182716,
 'od_wait': 50,
 'colsample_bylevel': 0.23949282816423725,
 'random_strength': 1,
 'l2_leaf_reg': 5.838687606767912,
 'max_depth': 4,
 'n_estimators': 872,
 'learning_rate': 0.026406880360521932}

In [15]:
best_model=CatBoostClassifier(**study.best_params)

best_model.fit(
    train,
    eval_set=eval,
    verbose=True
)

0:	learn: 0.6918988	test: 0.6921925	best: 0.6921925 (0)	total: 2.39ms	remaining: 2.08s
1:	learn: 0.6905442	test: 0.6914256	best: 0.6914256 (1)	total: 5.42ms	remaining: 2.36s
2:	learn: 0.6884376	test: 0.6897710	best: 0.6897710 (2)	total: 21.2ms	remaining: 6.13s
3:	learn: 0.6867599	test: 0.6884557	best: 0.6884557 (3)	total: 27ms	remaining: 5.86s
4:	learn: 0.6852289	test: 0.6873153	best: 0.6873153 (4)	total: 30ms	remaining: 5.2s
5:	learn: 0.6835582	test: 0.6858471	best: 0.6858471 (5)	total: 32.9ms	remaining: 4.75s
6:	learn: 0.6827744	test: 0.6852262	best: 0.6852262 (6)	total: 35.7ms	remaining: 4.41s
7:	learn: 0.6815354	test: 0.6842570	best: 0.6842570 (7)	total: 38.6ms	remaining: 4.17s
8:	learn: 0.6807678	test: 0.6838536	best: 0.6838536 (8)	total: 43.6ms	remaining: 4.18s
9:	learn: 0.6792948	test: 0.6826706	best: 0.6826706 (9)	total: 45.8ms	remaining: 3.95s
10:	learn: 0.6778102	test: 0.6815124	best: 0.6815124 (10)	total: 48.4ms	remaining: 3.79s
11:	learn: 0.6768120	test: 0.6814216	best: 0.6

## Submission

In [16]:
test.drop("ID", axis=1, inplace=True)

In [17]:
test[grp_1]=test[grp_1].fillna(0)
test[grp_2]=test[grp_2].apply(str).fillna(df.mode().iloc[0])
test[[x for x in grp_3 if x!="Target"]]=test[[x for x in grp_3 if x!="Target"]].fillna(df.mean())

In [18]:
ss.head()

,ID,Target
0,2661,1
1,2662,1
2,2663,1
3,2664,1
4,2665,1


In [19]:
predict = best_model.predict(test)
ss.Target = predict
ss.head()

,ID,Target
0,2661,1
1,2662,1
2,2663,0
3,2664,1
4,2665,1


In [20]:
ss.to_csv('submission_2.csv', index = False)